In [8]:
import sys
sys.path.append('../')
import time

import h5py
import epics
import numpy as np
import matplotlib.pyplot as plt

from siriuspy.devices import CAXCtrl
from caxscripts.h5file import HDF5File

In [9]:
CAX = CAXCtrl()

# Functions

Useful functions for the scan

In [10]:
MAXERRORCOUNT = 5

In [23]:

#todo: move to initial position

def current_parameters():
    return {
        'slit1': {
            'top': CAX.slit_A1.top_pos,
            'bottom': CAX.slit_A1.bottom_pos,
            'left': CAX.slit_A1.left_pos,
            'right': CAX.slit_A1.right_pos
        },
        'dvf1': {
            'expo_time': CAX.dvf_A1.acquisition_time
        },
        'dvf2': {
            'expo_time': CAX.dvf_A1.acquisition_time,
            'z_pos': CAX.dvf_B1.z_pos
        },
        'mirror': {
            'ry': CAX.mirror.ry_pos,
            'tx': CAX.mirror.tx_pos,
            'y1': CAX.mirror.y1_pos,
            'y2': CAX.mirror.y2_pos,
            'y3': CAX.mirror.y3_pos,
            'photocollector': CAX.mirror.photocurrent_signal
        }
    }

def move_all_slits(top, bottom, left, right):
    CAX.slit_A1.move_top(value=top)
    CAX.slit_A1.move_bottom(value=bottom)
    CAX.slit_A1.move_left(value=left)
    CAX.slit_A1.move_right(value=right)

def open_all_slits():
    # 1
    # 2: top=19.5, left=44.55, right=46.2
    move_all_slits(top=19.7-0.04,
                   bottom=35.8,
                   left=44.55,
                   right=47.2)

def move_robust_all_slits(top, bottom, left, right):
    CAX.slit_A1.move_robust_top(value=top)
    CAX.slit_A1.move_robust_bottom(value=bottom)
    CAX.slit_A1.move_robust_left(value=left)
    CAX.slit_A1.move_robust_right(value=right)

def get_image(dvf):

    count = 0
    while count < MAXERRORCOUNT:
        try:
            if not dvf.acquisition_status:
                dvf.cmd_acquire_on()
            return dvf.image
        except Exception as err:
            print(f" WARNING. When trying to fetch image from DVF1: {err} ")
            time.sleep(2)
            count += 1
            if count < MAXERRORCOUNT:
                print("\n Repeating the procedure...\n")
            else:
                raise Exception("Client exception")



In [24]:
open_all_slits()

# Scan

## Parameters

In [5]:
print('top:', CAX.slit_A1.top_pos,)
print('bottom:', CAX.slit_A1.bottom_pos,)
print('left:', CAX.slit_A1.left_pos,)
print('right:', CAX.slit_A1.right_pos)

top: 17.060000000000002
bottom: 34.1
left: 44.160000000000004
right: 45.5


In [7]:
move_all_slits(top=17+2,
               bottom=34,
               left=46,
               right=45.5)

In [12]:
top_pos_open = 19.7 - 0.04
bottom_pos_open = 35.8
left_pos_open = 44.6 - 0.04
right_pos_open = 47.2

rangex = 1.3 + 0.1
rangey = 4.8

proport = rangey/rangex

In [13]:
L = 0.4

Ndxs = 7
Ndys = int(proport*Ndxs)
dxs = np.linspace(0,rangex-L,Ndxs)
dys = np.linspace(0,rangey-L,Ndys)

In [14]:
print('N of points:',Ndys*Ndxs)
# print(f'scan time estimative: {Ndys*Ndxs*4/60:.3f} min')

N of points: 161


# Execution

Initial state

In [15]:
local_time = time.localtime()
formatted_time = time.strftime("%Y%m%d-%H%M%S", local_time)
formatted_date = time.strftime("%Y%m%d", local_time)

formatted_time, formatted_date

('20250904-162143', '20250904')

In [16]:
ANALYSIS = 'slit'

In [ ]:
parameters0 = current_parameters()

states_dir = '../states/'
state_name = f'{formatted_time}-{ANALYSIS}.txt'

with open(states_dir+state_name, 'w') as f:
    f.write(str(parameters0))

print(parameters0)

Initializate file

In [19]:
scaname = f'scan_{ANALYSIS}_{formatted_date}.h5'
datadir  = '/home/ids/data/'
direc    = f'{formatted_date}-Mirror-Slit-PC/'

In [20]:
file = HDF5File(filename=scaname, filedir=datadir+direc)

file.save_metadata(metadata_dict={
    'slit_top': CAX.slit_A1.top_pos,
    'slit_bottom': CAX.slit_A1.bottom_pos,
    'slit_left': CAX.slit_A1.left_pos,
    'slit_right': CAX.slit_A1.right_pos
})

Loop

In [22]:
open_all_slits()

In [21]:
t0 = time.time()

for i, dy in enumerate(dys):

    move_robust_all_slits(top    = top_pos_open-rangey+L+dy,
                          bottom = bottom_pos_open-dy,
                          left   = left_pos_open,
                          right  = right_pos_open-rangex+L)

    for j, dx in enumerate(dxs):
        print(f'scan step x {j+1}/{len(dxs)} and step y {i+1}/{len(dys)}')

        CAX.slit_A1.move_robust_left(value=left_pos_open-dx)
        CAX.slit_A1.move_robust_right(value=right_pos_open-rangex+L+dx)

        img1 = get_image(CAX.dvf_A1)
        expotime1 = CAX.dvf_A1.exposure_time
        img2 = get_image(CAX.dvf_B1)
        expotime2 = CAX.dvf_B1.exposure_time

        scan = f'scan-{i:03d}-{j:03d}'
        scanmetadata = {
            'slit_top'    : CAX.slit_A1.top_pos,
            'slit_bottom' : CAX.slit_A1.bottom_pos,
            'slit_left'   : CAX.slit_A1.left_pos,
            'slit_right'  : CAX.slit_A1.right_pos
        }
        file.save_group(grpname=scan, grpmetadata=scanmetadata)
        file.save_dataset(grpname=scan, dsetname=f'dvf1', dsetmetadata={'expo_time':expotime1}, dsetdata=img1)
        file.save_dataset(grpname=scan, dsetname=f'dvf2', dsetmetadata={'expo_time':expotime2}, dsetdata=img2)


t1 = time.time()

print()
print(f'ellapsed time [s]: {t1-t0}')

# move_all_slits(top    = top0,
#                bottom = bottom0,
#                left   = left0,
#                right  = right0)

Done!is Moving... | New pos: 15.26 | Curr: 15.26 | Dif: 0.076953125000001
Slit is Moving... | New pos: 35.8 | Curr: 35.76 | Dif: 0.03999999999999204
Posição atual: 35.7600
Done!is Moving... | New pos: 35.8 | Curr: 35.80 | Dif: 0.0
Done!
Done!is Moving... | New pos: 46.2 | Curr: 46.20 | Dif: 0.0
scan step x 1/7 and step y 1/23
Done!
Done!
scan step x 2/7 and step y 1/23
Done!is Moving... | New pos: 44.39333333333334 | Curr: 44.39 | Dif: 2.6041666664866625e-05
Done!is Moving... | New pos: 46.36666666666667 | Curr: 46.37 | Dif: 2.6041666664866625e-05
scan step x 3/7 and step y 1/23
Done!is Moving... | New pos: 44.22666666666667 | Curr: 44.23 | Dif: 2.6041666664866625e-05
Done!is Moving... | New pos: 46.53333333333334 | Curr: 46.53 | Dif: 2.6041666664866625e-05
scan step x 4/7 and step y 1/23


KeyboardInterrupt: 